In [40]:
import pandas as pd

# === 输入文件路径 ===
meer_path = "MeerKAT.csv"
fast_path = "FAST.csv"
output_path = "Combined_Pulsars.csv"

# === 读取两个CSV ===
meer_df = pd.read_csv(meer_path)
fast_df = pd.read_csv(fast_path)

# === 添加来源列（如果已有Telescope列，可以直接保留，不需要额外添加） ===
meer_df["Telescope"] = "MeerKAT"
fast_df["Telescope"] = "FAST"

# === 合并数据 ===
combined_df = pd.concat([meer_df, fast_df], ignore_index=True)

# === 保存 ===
combined_df.to_csv(output_path, index=False)

print(f"✅ 已生成合并文件: {output_path}")


✅ 已生成合并文件: Combined_Pulsars.csv


In [57]:
import pandas as pd
import urllib.parse
# ========= Settings =========
csv_path = "Combined_Pulsars.csv"   # Input CSV file
output_html = "index.html"  # Output HTML file

# ========= Formatting function =========
def format_value(r, val, up_err=None, low_err=None, sym_err=None, digits=1):
    """
    General error formatting:
    - If asymmetric errors (up_err, low_err) → value with top-right (+) & bottom-right (–) errors
    - If symmetric error (sym_err) → value ± err
    - Otherwise → value
    """
    try:
        value = f"{r[val]:.{digits}f}"
    except (ValueError, TypeError, KeyError):
        return ""

    # Asymmetric error → value with upper-right / lower-right errors
    if up_err and low_err and up_err in r and low_err in r:
        try:
            up = float(r[up_err])
            low = float(r[low_err])
            if not pd.isna(up) and not pd.isna(low) and (up != 0 or low != 0):
                return (
                    f'<span class="asym">'
                    f'  <span class="value">{value}</span>'
                    f'  <span class="up">+{up:.{digits}f}</span>'
                    f'  <span class="down">-{low:.{digits}f}</span>'
                    f'</span>'
                )
        except Exception:
            return f"{value}"

    # Symmetric error
    if sym_err and sym_err in r:
        try:
            e = float(r[sym_err])
            if not pd.isna(e) and e != 0:
                return f"{value} ± {e:.{digits}f}"
        except Exception:
            return f"{value}"

    # Default
    return value


# ========= Read data =========
df = pd.read_csv(csv_path)

# ========= 强制转换数值型 =========
for col in ["P0", "P-dot", "w10_center", "W10_MH", "L_I", "V_I", "absV_I", "PW", "all_alpha_range"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
# ========= Build formatted table =========
# ========= Build formatted table =========
# ========= Build formatted table =========
formatted_df = pd.DataFrame({

    "PSR": df["PSR"],

    "P (s)": df["P0"].map(lambda x: f"{x:.6f}" if pd.notna(x) else ""),
    "Ṗ (s/s)": df["P-dot"].map(lambda x: f"{x:.2e}" if pd.notna(x) else ""),

    "Telescope": df["Telescope"].astype(str),
    "Freq(MHz)": df["Freq(MHz)"].astype(str),

    "α₀ (°)": df.apply(
        lambda r: format_value(r, "best_alpha", "alpha_Upper_Error", "alpha_Lower_Error"),
        axis=1
    ),

    "β₀ (°)": df.apply(
        lambda r: format_value(r, "best_beta", "beta_upper_Error", "beta_lower_Error"),
        axis=1
    ),

    "φ₀ (°)": df.apply(
        lambda r: format_value(r, "best_phase_offset",
                               "phase_offset_upper_error",
                               "phase_offset_lower_error"),
        axis=1
    ),

    "ψ₀ (°)": df.apply(
        lambda r: format_value(r, "best_psi_offset",
                               "psi_offset_upper_error",
                               "psi_offset_lower_error"),
        axis=1
    ),

    # ===== χ² 不显示误差 =====
    "χ²": df["best_chi_red"].map(lambda x: f"{x:.1f}" if pd.notna(x) else ""),

    "K": df.apply(
        lambda r: format_value(r, "best_K", "K_upper_error", "K_lower_error"),
        axis=1
    ),

    "W10 (°)": df.apply(
        lambda r: format_value(r, "w10", sym_err="w10_error"),
        axis=1
    ),
    "W10_center": df["w10_center"].map(lambda x: f"{x:.1f}" if pd.notna(x) else ""),

    # ===== L/I 带误差 =====

    "L/I (%)": df.apply(
        lambda r: f"{r['L_I']*100:.1f} ± {r['σ_LI']*100:.1f}"
        if pd.notna(r["L_I"]) and pd.notna(r["σ_LI"]) else "",
        axis=1
    ),

    "V/I (%)": df.apply(
        lambda r: f"{r['V_I']*100:.1f} ± {r['σ_VI']*100:.1f}"
        if pd.notna(r["V_I"]) and pd.notna(r["σ_VI"]) else "",
        axis=1
    ),

    # ===== PDF 链接 =====
    "📄 2σ RVM": df.apply(
        lambda r: f'<a href="viewer.html?file={urllib.parse.quote("RVM_pulsars/" + r.PSR + "_" + r.Telescope + "_fiting.pdf")}" target="_blank">📄</a>',
        axis=1
    ),

    # ===== Plot 链接 =====
    "🖼 RVM Plot": df.apply(
        lambda r: f'<a href="viewer.html?file={urllib.parse.quote("RVM_pulsars/" + r.PSR + "_" + r.Telescope + "_RVMFIT.pdf")}" target="_blank">🖼</a>',
        axis=1
    )

})





# ========= Column order =========
col_order = [
    "PSR", "P (s)", "Ṗ (s/s)", "Telescope", "Freq(MHz)",
    "α₀ (°)", "β₀ (°)", "φ₀ (°)", "ψ₀ (°)",
    "χ²", "K",
    "W10 (°)", "W10_center", 
    "L/I (%)", "V/I (%)",
    "📄 2σ RVM", "🖼 RVM Plot"
]
# 强制列顺序
formatted_df = formatted_df.reindex(columns=col_order)

# ========= Generate HTML table =========
table_html = formatted_df.to_html(index=False, escape=False, border=0)

print(df[["L_I","σ_LI","V_I","σ_VI"]].head())

print(formatted_df[["L/I (%)", "V/I (%)"]].head())

        L_I      σ_LI       V_I      σ_VI
0  0.363335  0.001945  0.002528  0.001410
1  0.181454  0.000156 -0.002160  0.000116
2  0.179509  0.002270 -0.065520  0.001885
3  0.511558  0.000348 -0.115459  0.000274
4  0.525886  0.000345 -0.111950  0.000249
      L/I (%)      V/I (%)
0  36.3 ± 0.2    0.3 ± 0.1
1  18.1 ± 0.0   -0.2 ± 0.0
2  18.0 ± 0.2   -6.6 ± 0.2
3  51.2 ± 0.0  -11.5 ± 0.0
4  52.6 ± 0.0  -11.2 ± 0.0


In [59]:

    
    # 完整的HTML模板（与你提供的格式一致）
full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Pulsar Geometrical Parameters - Lei Hai</title>
  <link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
  <style>
    :root {{
      --neon-cyan: #00ffff;
      --neon-pink: #ff00ff;
      --neon-yellow: #ffff00;
      --neon-green: #00ff88;
      --accent: #ffe066;
      --accent2: #7dd3fc;
      --accent-light: rgba(255, 224, 102, 0.28);
      --accent-hover: rgba(255, 224, 102, 0.5);
      --text-subtle: #eaeaea;
      --panel: rgba(255,255,255,0.03);
      --panel-stroke: rgba(0,255,255,0.2);
      --soft-shadow: 0 15px 40px rgba(0,255,255,0.15);
    }}

    /* ===== 深空赛博主体 ===== */
    body {{
      margin: 0;
      font-family: "Arial", "Courier New", monospace;
      background: #000;
      color: white;
      overflow-x: hidden;
      position: relative;
    }}

    /* 多层立体星空（视差效果） */
    .stars-layer1, .stars-layer2, .stars-layer3 {{
      position: fixed;
      width: 200%;
      height: 200%;
      top: -50%;
      left: -50%;
      pointer-events: none;
    }}
    
    .stars-layer1 {{
      background-image: 
        radial-gradient(2px 2px at 20% 30%, white, transparent),
        radial-gradient(2px 2px at 60% 70%, white, transparent),
        radial-gradient(1px 1px at 90% 10%, white, transparent);
      background-size: 50% 50%;
      animation: drift 200s linear infinite;
      opacity: 0.5;
    }}
    
    .stars-layer2 {{
      background-image: 
        radial-gradient(3px 3px at 80% 10%, rgba(0,255,255,0.8), transparent),
        radial-gradient(3px 3px at 10% 90%, rgba(255,0,255,0.8), transparent);
      background-size: 75% 75%;
      animation: drift 150s linear infinite reverse;
      opacity: 0.3;
    }}
    
    .stars-layer3 {{
      background-image: 
        radial-gradient(1px 1px at 50% 50%, white, transparent);
      background-size: 25% 25%;
      animation: drift 100s linear infinite;
      opacity: 0.8;
    }}
    
    @keyframes drift {{
      0% {{ transform: translate3d(0, 0, 0) rotate(0deg); }}
      100% {{ transform: translate3d(100px, 100px, 0) rotate(360deg); }}
    }}

    /* 磁场网格动画 */
    .magnetic-grid {{
      position: fixed;
      width: 100%;
      height: 100%;
      background-image: 
        linear-gradient(rgba(0,255,255,0.1) 1px, transparent 1px),
        linear-gradient(90deg, rgba(0,255,255,0.1) 1px, transparent 1px),
        linear-gradient(rgba(255,0,255,0.05) 2px, transparent 2px),
        linear-gradient(90deg, rgba(255,0,255,0.05) 2px, transparent 2px);
      background-size: 50px 50px, 50px 50px, 100px 100px, 100px 100px;
      animation: gridMove 20s linear infinite;
      pointer-events: none;
      z-index: 1;
    }}
    
    @keyframes gridMove {{
      0% {{ transform: perspective(800px) rotateX(25deg) translateY(0); }}
      100% {{ transform: perspective(800px) rotateX(25deg) translateY(-100px); }}
    }}

    /* 扫描线效果 */
    .scan-lines {{
      position: fixed;
      width: 100%;
      height: 100%;
      background: linear-gradient(
        0deg,
        transparent 48%,
        rgba(0,255,255,0.03) 50%,
        transparent 52%
      );
      background-size: 100% 4px;
      animation: scanMove 8s linear infinite;
      pointer-events: none;
      z-index: 2;
    }}
    
    .scan-line-horizontal {{
      position: fixed;
      width: 100%;
      height: 2px;
      background: linear-gradient(90deg, transparent, rgba(0,255,255,0.8), transparent);
      animation: scanHorizontal 4s linear infinite;
      pointer-events: none;
      z-index: 3;
    }}
    
    @keyframes scanMove {{
      0% {{ background-position: 0 0; }}
      100% {{ background-position: 0 10px; }}
    }}
    
    @keyframes scanHorizontal {{
      0% {{ top: -2px; }}
      100% {{ top: 100%; }}
    }}

    /* 主容器 */
    .overlay {{
      background: linear-gradient(180deg, 
        rgba(0,0,0,0.4) 0%, 
        rgba(0,20,40,0.6) 50%, 
        rgba(0,0,0,0.8) 100%);
      min-height: 100vh;
      padding: 48px 24px 40px;
      position: relative;
      z-index: 10;
      backdrop-filter: blur(2px);
    }}
    
    .container {{
      max-width: 1100px;
      margin: 0 auto;
      text-align: center;
    }}

    /* ===== 霓虹标题（静态渐变） ===== */
    h1 {{
      font-size: 3em;
      margin: 0 0 20px;
      background: linear-gradient(90deg, 
        var(--neon-cyan), 
        var(--neon-pink), 
        var(--neon-yellow), 
        var(--neon-green),
        var(--neon-cyan));
      background-size: 200% auto;
      background-position: 0% center;
      -webkit-background-clip: text;
      background-clip: text;
      -webkit-text-fill-color: transparent;
      text-transform: uppercase;
      letter-spacing: 3px;
      text-shadow: 
        0 0 20px rgba(0,255,255,0.5),
        0 0 40px rgba(255,0,255,0.3),
        0 0 60px rgba(0,255,255,0.2);
      position: relative;
    }}
    
    h1::before {{
      content: attr(data-text);
      position: absolute;
      top: 0;
      left: 0;
      width: 100%;
      height: 100%;
      background: linear-gradient(90deg, 
        var(--neon-cyan), 
        var(--neon-pink), 
        var(--neon-yellow), 
        var(--neon-green),
        var(--neon-cyan));
      background-size: 200% auto;
      background-position: 0% center;
      -webkit-background-clip: text;
      background-clip: text;
      -webkit-text-fill-color: transparent;
      filter: blur(3px);
      opacity: 0.5;
      z-index: -1;
    }}

    .divider {{
      width: 300px;
      height: 3px;
      border: 0;
      margin: 20px auto 30px;
      background: linear-gradient(90deg, 
        transparent, 
        var(--neon-cyan), 
        var(--neon-pink), 
        var(--neon-cyan),
        transparent);
      box-shadow: 
        0 0 20px rgba(0,255,255,0.5),
        0 0 40px rgba(255,0,255,0.3);
      animation: dividerGlow 2s ease-in-out infinite;
    }}
    
    @keyframes dividerGlow {{
      0%, 100% {{ opacity: 0.6; transform: scaleX(1); }}
      50% {{ opacity: 1; transform: scaleX(1.05); }}
    }}

    /* 内容区域 */
    .content-block {{
      max-width: 1000px;
      margin: 0 auto;
      text-align: center;
    }}
    
    .physics-desc {{
      margin: 18px auto 28px;
      max-width: 1000px;
      font-size: 1em;
      line-height: 1.85;
      color: var(--text-subtle);
      text-align: justify;
      text-shadow: 0 0 5px rgba(0,255,255,0.1);
      background: linear-gradient(135deg, 
        rgba(0,255,255,0.02), 
        rgba(255,0,255,0.02));
      padding: 15px;
      border-radius: 10px;
      border: 1px solid rgba(0,255,255,0.1);
    }}

    /* ===== 图片效果（取消点击） ===== */
    .figure-box img {{
      width: 85%;
      max-width: 85%;
      max-height: 600px;
      border-radius: 15px;
      display: block;
      margin: 0 auto;
      box-shadow: 
        0 0 30px rgba(0,255,255,0.3),
        0 0 60px rgba(255,0,255,0.1);
      border: 2px solid rgba(0,255,255,0.3);
    }}
    
    .figure-caption {{
      margin: 15px 0 25px;
      font-size: 0.95em;
      color: var(--neon-cyan);
      font-style: italic;
      opacity: 0.9;
      text-shadow: 0 0 10px rgba(0,255,255,0.5);
    }}

    /* ===== 全息下载卡片 ===== */
    .toolbar {{
      position: relative;
      text-align: center;
      margin: 30px auto 20px;
      padding: 25px;
      max-width: 980px;
      border-radius: 20px;
      overflow: hidden;
      isolation: isolate;
      animation: holoBreathe 4s ease-in-out infinite;
      background: rgba(0,0,0,0.6);
    }}
    
    @keyframes holoBreathe {{
      0%, 100% {{ transform: scale(1); }}
      50% {{ transform: scale(1.01); }}
    }}
    
    .toolbar::before {{
      content: "";
      position: absolute;
      inset: -3px;
      border-radius: 20px;
      background: conic-gradient(from 0deg,
        var(--neon-cyan), 
        var(--neon-pink), 
        var(--neon-yellow), 
        var(--neon-green),
        var(--neon-cyan));
      animation: holoRotate 3s linear infinite;
      z-index: -2;
    }}
    
    .toolbar::after {{
      content: "";
      position: absolute;
      inset: 2px;
      border-radius: 18px;
      background: linear-gradient(135deg, 
        rgba(0,0,0,0.9), 
        rgba(0,20,40,0.9));
      z-index: -1;
    }}
    
    @keyframes holoRotate {{
      0% {{ transform: rotate(0deg); filter: hue-rotate(0deg); }}
      100% {{ transform: rotate(360deg); filter: hue-rotate(360deg); }}
    }}
    
    .toolbar-title {{
      display: block;
      font-size: 1.2em;
      font-weight: 800;
      background: linear-gradient(90deg, var(--neon-cyan), var(--neon-pink));
      -webkit-background-clip: text;
      background-clip: text;
      -webkit-text-fill-color: transparent;
      margin-bottom: 10px;
      letter-spacing: 2px;
      text-transform: uppercase;
      animation: titleFlicker 2s ease-in-out infinite;
    }}
    
    @keyframes titleFlicker {{
      0%, 100% {{ opacity: 1; }}
      50% {{ opacity: 0.8; }}
    }}
    
    .toolbar-desc {{
      font-size: 0.92em;
      color: #e9e9e9;
      margin: 0 auto 15px;
      max-width: 720px;
      line-height: 1.6;
      opacity: 0.9;
    }}
    
    .download-btn {{
      display: inline-block;
      margin: 8px;
      padding: 12px 25px;
      color: #000;
      background: linear-gradient(135deg, var(--neon-cyan), var(--neon-green));
      border: none;
      border-radius: 30px;
      text-decoration: none;
      font-weight: 700;
      text-transform: uppercase;
      letter-spacing: 1px;
      transition: all 0.3s ease;
      box-shadow: 
        0 0 20px rgba(0,255,255,0.3),
        inset 0 0 20px rgba(255,255,255,0.1);
      position: relative;
      overflow: hidden;
    }}
    
    .download-btn::before {{
      content: "";
      position: absolute;
      top: -50%;
      left: -50%;
      width: 200%;
      height: 200%;
      background: linear-gradient(45deg, 
        transparent, 
        rgba(255,255,255,0.3), 
        transparent);
      transform: rotate(45deg);
      transition: all 0.5s;
      opacity: 0;
    }}
    
    .download-btn:hover {{
      transform: translateY(-3px) scale(1.05);
      color: #fff;
      background: linear-gradient(135deg, var(--neon-pink), var(--neon-cyan));
      box-shadow: 
        0 0 30px rgba(255,0,255,0.5),
        0 10px 30px rgba(0,0,0,0.3),
        inset 0 0 30px rgba(255,255,255,0.2);
    }}
    
    .download-btn:hover::before {{
      animation: btnShine 0.5s;
    }}
    
    @keyframes btnShine {{
      0% {{ left: -100%; opacity: 0; }}
      50% {{ opacity: 1; }}
      100% {{ left: 100%; opacity: 0; }}
    }}

    /* ===== 简化后的表格样式 ===== */
    .table-wrapper {{
      margin-top: 20px;
      overflow-x: auto;
      background: rgba(0,0,0,0.4);
      border: 1px solid rgba(0,255,255,0.3);
      border-radius: 15px;
      padding: 10px;
      box-shadow: 0 0 20px rgba(0,255,255,0.1);
      position: relative;
    }}
    
    table {{
      border-collapse: collapse;
      width: 100%;
      white-space: nowrap;
      border-radius: 10px;
      overflow: hidden;
    }}
    
    th, td {{
      padding: 12px 14px;
      border: 1px solid rgba(0,255,255,0.15);
      font-size: 0.92em;
      text-align: center !important;
    }}
    
    /* 简洁的粘性表头 */
    thead th {{
      position: sticky;
      top: 0;
      z-index: 100;
      background: rgba(0,100,120,0.3);
      color: #fff;
      font-weight: 700;
      letter-spacing: 0.5px;
      backdrop-filter: blur(10px);
      border-bottom: 2px solid rgba(0,255,255,0.4);
    }}
    
    /* 简单的斑马纹 */
    tbody tr:nth-child(odd) td {{
      background: rgba(0,40,60,0.15);
    }}
    
    tbody tr:nth-child(even) td {{
      background: rgba(0,20,40,0.15);
    }}
    
    /* 基础文字样式 */
    td {{
      color: #ddd;
    }}

    /* ===== DataTables 霓虹控件 ===== */
    .dataTables_wrapper {{
      color: var(--neon-cyan);
    }}
    
    .dataTables_wrapper .dataTables_filter input,
    .dataTables_wrapper .dataTables_length select {{
      background: rgba(0,0,0,0.6);
      border: 2px solid var(--neon-cyan);
      color: var(--neon-cyan);
      border-radius: 20px;
      padding: 8px 15px;
      outline: none;
      transition: all 0.3s ease;
      box-shadow: 
        0 0 10px rgba(0,255,255,0.2),
        inset 0 0 10px rgba(0,255,255,0.1);
    }}
    
    .dataTables_wrapper .dataTables_filter input:focus,
    .dataTables_wrapper .dataTables_length select:focus {{
      border-color: var(--neon-pink);
      box-shadow: 
        0 0 20px rgba(255,0,255,0.4),
        inset 0 0 15px rgba(255,0,255,0.2);
      transform: scale(1.05);
    }}
    
    .dataTables_wrapper .dataTables_paginate .paginate_button {{
      color: var(--neon-cyan) !important;
      border: 2px solid rgba(0,255,255,0.3) !important;
      background: linear-gradient(135deg, 
        rgba(0,0,0,0.6), 
        rgba(0,20,40,0.6)) !important;
      border-radius: 20px !important;
      margin: 0 3px !important;
      padding: 6px 12px !important;
      transition: all 0.3s ease !important;
      text-shadow: 0 0 5px rgba(0,255,255,0.5);
    }}
    
    .dataTables_wrapper .dataTables_paginate .paginate_button:hover {{
      border-color: var(--neon-pink) !important;
      background: linear-gradient(135deg, 
        rgba(255,0,255,0.2), 
        rgba(0,255,255,0.2)) !important;
      transform: scale(1.1);
      box-shadow: 
        0 0 15px rgba(255,0,255,0.4),
        inset 0 0 10px rgba(255,0,255,0.2);
    }}
    
    .dataTables_wrapper .dataTables_paginate .paginate_button.current {{
      border-color: var(--neon-yellow) !important;
      background: linear-gradient(135deg, 
        rgba(255,255,0,0.3), 
        rgba(255,255,0,0.1)) !important;
      color: #fff !important;
      box-shadow: 
        0 0 20px rgba(255,255,0,0.5),
        inset 0 0 15px rgba(255,255,0,0.3) !important;
      animation: currentPulse 2s ease-in-out infinite;
    }}
    
    @keyframes currentPulse {{
      0%, 100% {{ transform: scale(1); }}
      50% {{ transform: scale(1.05); }}
    }}
    
    .dataTables_wrapper .dataTables_info {{
      opacity: 0.9;
      margin-top: 10px;
      color: var(--neon-cyan);
      text-shadow: 0 0 5px rgba(0,255,255,0.3);
    }}

    /* ===== 误差表达式样式 ===== */
    .asym {{
      display: inline-grid;
      grid-template-columns: auto auto;
      grid-template-rows: auto auto;
      column-gap: 0.2ch;
      align-items: center;
    }}
    .asym .value {{ 
      grid-row: 1 / span 2; 
      grid-column: 1; 
      color: #fff; 
    }}
    .asym .up {{ 
      grid-row: 1; 
      grid-column: 2; 
      font-size: 0.72em; 
      align-self: end; 
      color: #fff; 
    }}
    .asym .down {{ 
      grid-row: 2; 
      grid-column: 2; 
      font-size: 0.72em; 
      align-self: start; 
      color: #fff; 
    }}
    .sym {{ 
      display: inline-flex; 
      align-items: baseline; 
      gap: .25ch; 
    }}
    .sym .pm {{ 
      font-size: .85em; 
      opacity: .9; 
    }}
    .sym .err {{ 
      font-size: .85em; 
      opacity: .95; 
    }}

    /* 参数说明样式 */
    .param-desc {{
      margin: 20px 0 30px;
      font-size: 0.95em;
      line-height: 1.85;
      color: var(--text-subtle);
      text-align: justify;
      padding: 20px;
      background: linear-gradient(135deg, 
        rgba(0,255,255,0.02), 
        rgba(255,0,255,0.02));
      border: 1px solid rgba(0,255,255,0.2);
      border-radius: 10px;
      position: relative;
    }}
    
    .param-desc h3 {{
      color: var(--neon-cyan);
      text-shadow: 0 0 10px rgba(0,255,255,0.5);
      margin-bottom: 15px;
    }}
    
    .param-desc b {{
      color: var(--neon-yellow);
      text-shadow: 0 0 5px rgba(255,255,0,0.5);
    }}

    /* 响应式和动画优化 */
    @media (prefers-reduced-motion: reduce) {{
      *, *::before, *::after {{
        animation: none !important;
        transition: none !important;
      }}
    }}
  </style>

  <script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>
  <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
  <script>
    function formatErrorCells(scope) {{
      const asymRe = /^\\s*(-?\\d+(?:\\.\\d+)?(?:[eE][+-]?\\d+)?)\\s*\\+(\\d+(?:\\.\\d+)?(?:[eE][+-]?\\d+)?)\\s*-(\\d+(?:\\.\\d+)?(?:[eE][+-]?\\d+)?)\\s*$/;
      const symRe  = /^\\s*(-?\\d+(?:\\.\\d+)?(?:[eE][+-]?\\d+)?)\\s*[±\\u00B1]\\s*(\\d+(?:\\.\\d+)?(?:[eE][+-]?\\d+)?)\\s*$/;
      $(scope).find('tbody td').each(function () {{
        const $td = $(this);
        const txt = $td.text().trim();
        if ($td.find('.asym,.sym').length) return;
        let m = txt.match(asymRe);
        if (m) {{
          const v = m[1], up = m[2], dn = m[3];
          $td.html(`<span class="asym"><span class="value">${{v}}</span><span class="up">+${{up}}</span><span class="down">-${{dn}}</span></span>`);
          return;
        }}
        m = txt.match(symRe);
        if (m) {{
          const v = m[1], e = m[2];
          $td.html(`<span class="sym"><span class="value">${{v}}</span><span class="pm">±</span><span class="err">${{e}}</span></span>`);
        }}
      }});
    }}

    $(document).ready(function() {{
      $('h1').attr('data-text', $('h1').text());
      
      const table = $('table').DataTable({{
        pageLength: 50,
        lengthMenu: [20, 50, 100, 200, 500],
        ordering: true,
        searching: true,
        scrollX: true,
        autoWidth: true
      }});
      
      formatErrorCells(table.table().container());
      table.on('draw', () => formatErrorCells(table.table().container()));
    }});
  </script>
</head>
<body>
  <div class="stars-layer1"></div>
  <div class="stars-layer2"></div>
  <div class="stars-layer3"></div>
  <div class="magnetic-grid"></div>
  <div class="scan-lines"></div>
  <div class="scan-line-horizontal"></div>

  <div class="overlay">
    <div class="container">
      <h1>Pulsar Geometrical Parameters</h1>
      <div class="divider"></div>

      <div class="physics-desc">
        &emsp;&emsp; The Rotating Vector Model (RVM) provides a geometrical framework for studying pulsar polarization, linking the observed variation of the linear polarization position angle (PA) across pulse phase to the geometrical relationship between the pulsar’s magnetic axis and the observer’s line of sight. In this model, the magnetic inclination angle (α) describes the tilt of the magnetic axis relative to the rotation axis, while the impact angle (β) characterizes the closest approach of the line of sight to the magnetic axis. According to the geometrical emission theory of electromagnetic radiation, the morphology of the PA curve is governed by the structure of open magnetic field lines in the magnetosphere. By fitting the RVM to observational data, one can constrain these geometrical parameters and further infer the beam opening angle, emission height, and possible propagation effects within the magnetosphere.
      <br><br>
        &emsp;&emsp;   With the rapid development of large radio telescopes such as FAST and MeerKAT, together with high-sensitivity observation programs including the FAST Galactic Plane Pulsar Snapshot survey (GPPS) and the MeerKAT Thousand Pulsar Array (TPA), the number of pulsars with polarization measurements has increased significantly, providing an unprecedentedly large dataset for systematic RVM-fitting studies. These observations not only deepen our understanding of pulsar geometry but also provide valuable physical clues for investigating the plasma environment in pulsar magnetospheres and the evolution of pulsar populations. Below, we present the sky distribution and geometrical parameters of RVM-fitted pulsars based on observations from FAST and MeerKAT.
       </div>

      <div class="content-block">
        <div class="figure-box">
          <img src="RVM_pulsars/pulsar_mollweide_projection.png" alt="Pulsar Geometry">
        </div>
        <div class="figure-caption">Figure. Distribution of pulsars in equatorial coordinates (RA/Dec).</div>

        <div class="param-desc">
          <h3>Table Parameters</h3>
          <p>
            <b>PSR</b> — Pulsar name (J2000 designation);
            <b>P (s)</b> — Spin period in seconds;
            <b>Ṗ (s/s)</b> — Period derivative;
            <b>Telescope</b> — Observation telescope (MeerKAT / FAST / etc.);
            <b>Freq(MHz)</b> — Observing frequency band (MHz);
            <b>α₀ (°)</b> — Magnetic inclination angle;
            <b>β₀ (°)</b> — Impact angle;
            <b>φ₀ (°)</b> — Phase of steepest gradient;
            <b>ψ₀ (°)</b> — Reference position angle;
            <b>χ²</b> — Reduced chi-squared;
            <b>K</b> — Maximum slope of the RVM curve at φ₀ (steepest gradient);
            <b>W10 (°)</b> — Pulse width at 10% peak;
            <b>W10_center</b> — Central phase of W10;
            <b>L/I(%)</b> — Fractional linear polarization;
            <b>V/I(%)</b> — Fractional circular polarizatio
            <b>📄 2σ RVM</b> — RVM curves within the 2σ confidence interval;
            <b>🖼 RVM Plot</b> — Pulse profile and PA curve with best-fit RVM.
          </p>
        </div>

        <div class="table-wrapper">
          {table_html}
        </div>

        <div class="toolbar">
          <span class="toolbar-title">Download Data</span>
          <p class="toolbar-desc">
          You can download the RVM-fitted pulsar parameter tables derived from observations with the <b>FAST</b> and <b>MeerKAT</b> telescopes. The datasets are provided in <b>CSV format</b> for further quantitative analysis, comparison, or model fitting. If you have any questions, feel free to contact us at <a href="mailto:leihai5514@gmail.com" style="color:#FFFF00;">leihai5514@gmail.com</a>.
          </p>


          <a href="table/GPTS.csv" download class="download-btn">⬇️ GPTS.csv</a>
          <a href="table/TPAS.csv" download class="download-btn">⬇️ TPAS.csv</a>
        </div>

      </div>
    </div>
  </div>
</body>
</html>
"""
    
    # 写入文件
with open(output_html, "w", encoding="utf-8") as f:
     f.write(full_html)
